# RAG Evaluations

This notebook evaluates a simple RAG chatbot over Revolut help articles using synthetic customer queries and binary LLM judges.

## Pipeline Overview

1. **Seeds**: 10 personas × 25 problems × 6 modifiers = 1,500 synthetic queries
2. **Generation**: LLM writes customer messages following persona/problem/modifier
3. **Vibe Check**: Validates queries for realism (human-mobile patterns)
4. **RAG Pipeline**: NumPy-based retrieval + LLM answer generation
5. **Six Binary Judges**: relevance, groundedness, completeness, actionability, tone/empathy, safety/compliance

## Setup and Data Loading

**Column Dictionary:**
- `persona_id`: seed persona identifier
- `problem_id`: seed problem identifier  
- `modifier_id`: seed modifier identifier
- `query`: generated customer message
- `answer`: RAG-generated response
- `{criterion}_passed`: boolean judgment (6 criteria)
- `{criterion}_reasoning`: judge explanation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Load data with absolute paths
base_dir = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
data_dir = base_dir / 'data' / 'outputs'

validated_df = pd.read_csv(data_dir / 'validated_queries.csv')
rag_df = pd.read_csv(data_dir / 'rag_outputs.csv')
eval_df = pd.read_csv(data_dir / 'eval_results.csv')

print(f"Validated queries: {len(validated_df)}")
print(f"RAG outputs: {len(rag_df)}")
print(f"Final evaluations: {len(eval_df)}")

## Pipeline Flow Summary

**Takeaway:** 1,500 queries generated → 1,080 passed validation (72% acceptance) → 1,080 RAG responses → 1,080 evaluated by all 6 judges.

In [ ]:
# Pipeline flow visualization
stages = {
    'Generated': 1500,
    'Validated': len(validated_df),
    'Passed Vibe Check': validated_df['passed'].sum(),
    'RAG Outputs': len(rag_df),
    'Evaluated': len(eval_df)
}

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(len(stages))
bars = ax.bar(x_pos, stages.values(), color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
ax.set_xticks(x_pos)
ax.set_xticklabels(stages.keys(), rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('RAG Evaluation Pipeline Flow')
plt.tight_layout()
plt.show()

print("\nPipeline Summary:")
for stage, count in stages.items():
    print(f"  {stage}: {count}")

## Individual Judge Performance

**Takeaway:** Strongest on relevance (78%) and completeness (74%). Critical weakness in groundedness (19%) indicating hallucinations, and near-zero tone/empathy (0.37%).

In [ ]:
# Calculate pass rates per judge
criteria = ['relevance', 'groundedness', 'completeness', 'actionability', 'tone_empathy', 'safety_compliance']
pass_rates = {}
for criterion in criteria:
    col = f'{criterion}_passed'
    pass_rates[criterion] = eval_df[col].mean()

# Sort by pass rate
sorted_rates = sorted(pass_rates.items(), key=lambda x: x[1], reverse=True)

# Create pass rate table
pass_df = pd.DataFrame([{
    'Judge': judge,
    'Pass Rate': f"{rate:.2%}",
    'Passed': int(eval_df[f'{judge}_passed'].sum()),
    'Failed': len(eval_df) - int(eval_df[f'{judge}_passed'].sum())
} for judge, rate in sorted_rates])

display(pass_df)

# Create barplot
fig, ax = plt.subplots(figsize=(10, 5))
judges = [j for j, _ in sorted_rates]
rates = [r for _, r in sorted_rates]
colors = ['#2ecc71' if r > 0.5 else '#e74c3c' if r < 0.3 else '#f39c12' for r in rates]
bars = ax.barh(judges, rates, color=colors)
ax.set_xlabel('Pass Rate')
ax.set_title('Individual Judge Pass Rates')
ax.set_xlim(0, 1)
for bar, rate in zip(bars, rates):
    width = bar.get_width()
    ax.text(width + 0.02, bar.get_y() + bar.get_height()/2, f'{rate:.1%}', 
            ha='left', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print("\nIndividual Judge Performance:")
for judge, rate in sorted_rates:
    status = "✓ Strong" if rate > 0.7 else "⚠️ Weak" if rate < 0.4 else "~ Moderate"
    print(f"  {judge}: {rate:.2%} {status}")

## Overall Performance (All Judges)

**Takeaway:** 0% overall pass rate (no query passed all 6 judges). This is driven by tone_empathy at 0.37% and groundedness at 19.26% - product-level issues requiring prompt engineering or judge recalibration.

In [ ]:
# Calculate overall pass rate (all 6 judges)
all_passed = eval_df[[f'{c}_passed' for c in criteria]].all(axis=1)
overall_pass_rate = all_passed.mean()

# Count how many judges each query passed
judges_passed = eval_df[[f'{c}_passed' for c in criteria]].sum(axis=1)

# Distribution of judges passed
dist_df = pd.DataFrame({
    'Judges Passed': judges_passed.value_counts().sort_index().index,
    'Count': judges_passed.value_counts().sort_index().values,
    'Percentage': (judges_passed.value_counts().sort_index().values / len(eval_df) * 100).round(1)
})

display(dist_df)

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Overall pass rate
ax1.bar(['Overall Pass Rate'], [overall_pass_rate], color='#e74c3c', alpha=0.7)
ax1.set_ylabel('Pass Rate')
ax1.set_title('Overall Pass Rate (All 6 Judges)')
ax1.set_ylim(0, 0.1)
ax1.text(0, overall_pass_rate + 0.005, f'{overall_pass_rate:.2%}', ha='center', fontsize=12, fontweight='bold')

# Distribution of judges passed
ax2.bar(dist_df['Judges Passed'], dist_df['Count'], color='#3498db', alpha=0.7)
ax2.set_xlabel('Number of Judges Passed')
ax2.set_ylabel('Count')
ax2.set_title('Distribution: How Many Judges Each Query Passed')
for i, (judges, count) in enumerate(zip(dist_df['Judges Passed'], dist_df['Count'])):
    ax2.text(judges, count + 10, str(count), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nOverall Pass Rate (all 6 judges): {overall_pass_rate:.2%}")
print(f"Perfect passes (6/6): {all_passed.sum()} / {len(eval_df)}")
print(f"Average judges passed per query: {judges_passed.mean():.2f} / 6")

## Performance by Target Thresholds

**Takeaway:** Only relevance (78%) meets the 70% threshold. Actionability (68%) misses narrowly. Groundedness (19%), safety (38%), and tone (0%) are critical misses. Tone_empathy at 0.37% is likely over-strict (calibration issue).

In [ ]:
# Define target thresholds
targets = {
    'relevance': 0.70,
    'groundedness': 0.70,
    'completeness': 0.70,
    'actionability': 0.70,
    'tone_empathy': 0.50,
    'safety_compliance': 0.60
}

# Create threshold analysis
threshold_df = pd.DataFrame([{
    'Judge': judge,
    'Actual': f"{pass_rates[judge]:.2%}",
    'Target': f"{target:.0%}",
    'Gap': f"{(pass_rates[judge] - target):.2%}",
    'Status': '✓ Pass' if pass_rates[judge] >= target else '❌ Miss'
} for judge, target in targets.items()])

display(threshold_df)

# Visualize gaps
fig, ax = plt.subplots(figsize=(10, 6))
judges = list(targets.keys())
actuals = [pass_rates[j] for j in judges]
target_vals = list(targets.values())
x = np.arange(len(judges))
width = 0.35

bars1 = ax.bar(x - width/2, actuals, width, label='Actual', color='#3498db')
bars2 = ax.bar(x + width/2, target_vals, width, label='Target', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(judges, rotation=45, ha='right')
ax.set_ylabel('Pass Rate')
ax.set_title('Performance vs Target Thresholds')
ax.legend()
ax.set_ylim(0, 1)

for bar1, bar2, actual, target in zip(bars1, bars2, actuals, target_vals):
    gap = actual - target
    if gap >= 0:
        ax.text(bar1.get_x() + bar1.get_width()/2, bar1.get_height() + 0.02, 
                f'+{gap:.1%}', ha='center', fontsize=9, color='#2ecc71')
    else:
        ax.text(bar2.get_x() + bar2.get_width()/2, bar2.get_height() + 0.02, 
                f'{gap:.1%}', ha='center', fontsize=9, color='#e74c3c')

plt.tight_layout()
plt.show()

misses = [j for j in judges if pass_rates[j] < targets[j]]
print(f"\nThreshold Analysis:")
print(f"  Passing: {len(judges) - len(misses)}/{len(judges)} judges")
print(f"  Missing targets: {', '.join(misses)}")

## Performance by Persona

**Takeaway:** Performance varies across personas but overall pass rates remain near 0% due to tone/groundedness failures. Some personas show slightly better actionability and safety scores.

In [ ]:
# Calculate persona performance
persona_perf = []
for persona in eval_df['persona_id'].unique():
    subset = eval_df[eval_df['persona_id'] == persona]
    persona_pass = subset[[f'{c}_passed' for c in criteria]].all(axis=1).mean()
    perf = {
        'persona': persona,
        'count': len(subset),
        'overall_pass': persona_pass,
        'relevance': subset['relevance_passed'].mean(),
        'groundedness': subset['groundedness_passed'].mean(),
        'completeness': subset['completeness_passed'].mean(),
        'actionability': subset['actionability_passed'].mean(),
        'tone_empathy': subset['tone_empathy_passed'].mean(),
        'safety': subset['safety_compliance_passed'].mean()
    }
    persona_perf.append(perf)

persona_df = pd.DataFrame(persona_perf).sort_values('overall_pass', ascending=False)
display(persona_df.head(10))

# Create heatmap
heatmap_data = persona_df.set_index('persona')[['relevance', 'groundedness', 'completeness', 
                                                  'actionability', 'tone_empathy', 'safety']]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heatmap_data * 100, annot=True, fmt='.1f', cmap='RdYlGn', 
            vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Pass Rate (%)'})
ax.set_title('Judge Pass Rates by Persona (%)')
ax.set_ylabel('Persona')
ax.set_xlabel('Judge')
plt.tight_layout()
plt.show()

print("\nWorst-performing personas (overall pass rate):")
for _, row in persona_df.tail(3).iterrows():
    print(f"  {row['persona']}: {row['overall_pass']:.2%} ({row['count']} queries)")

## Performance by Problem Type

**Takeaway:** All problem types show similar patterns - strong relevance/completeness but weak groundedness and near-zero tone. Card/transfer issues tend to have slightly better safety scores.

In [ ]:
# Calculate problem performance
problem_perf = []
for problem in eval_df['problem_id'].unique():
    subset = eval_df[eval_df['problem_id'] == problem]
    if len(subset) >= 10:  # Only include problems with >= 10 queries
        problem_pass = subset[[f'{c}_passed' for c in criteria]].all(axis=1).mean()
        perf = {
            'problem': problem,
            'count': len(subset),
            'overall_pass': problem_pass,
            'relevance': subset['relevance_passed'].mean(),
            'groundedness': subset['groundedness_passed'].mean(),
            'completeness': subset['completeness_passed'].mean(),
            'actionability': subset['actionability_passed'].mean(),
            'tone_empathy': subset['tone_empathy_passed'].mean(),
            'safety': subset['safety_compliance_passed'].mean()
        }
        problem_perf.append(perf)

problem_df = pd.DataFrame(problem_perf).sort_values('overall_pass', ascending=False)
display(problem_df.head(10))

# Create heatmap for top 15 problems
top_problems = problem_df.head(15)
heatmap_data = top_problems.set_index('problem')[['relevance', 'groundedness', 'completeness', 
                                                  'actionability', 'tone_empathy', 'safety']]
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(heatmap_data * 100, annot=True, fmt='.1f', cmap='RdYYlGn', 
            vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Pass Rate (%)'})
ax.set_title('Judge Pass Rates by Problem Type (Top 15, %)')
ax.set_ylabel('Problem')
ax.set_xlabel('Judge')
plt.tight_layout()
plt.show()

print("\nWorst-performing problems (overall pass rate, n>=10):")
for _, row in problem_df.tail(5).iterrows():
    print(f"  {row['problem']}: {row['overall_pass']:.2%} ({row['count']} queries)")

## Performance by Modifier

**Takeaway:** Non-native and urgent modifiers show slightly different patterns but all suffer from the same groundedness and tone issues.

In [ ]:
# Calculate modifier performance
modifier_perf = []
for modifier in eval_df['modifier_id'].unique():
    subset = eval_df[eval_df['modifier_id'] == modifier]
    modifier_pass = subset[[f'{c}_passed' for c in criteria]].all(axis=1).mean()
    perf = {
        'modifier': modifier,
        'count': len(subset),
        'overall_pass': modifier_pass,
        'relevance': subset['relevance_passed'].mean(),
        'groundedness': subset['groundedness_passed'].mean(),
        'completeness': subset['completeness_passed'].mean(),
        'actionability': subset['actionability_passed'].mean(),
        'tone_empathy': subset['tone_empathy_passed'].mean(),
        'safety': subset['safety_compliance_passed'].mean()
    }
    modifier_perf.append(perf)

modifier_df = pd.DataFrame(modifier_perf).sort_values('overall_pass', ascending=False)
display(modifier_df)

# Create heatmap
heatmap_data = modifier_df.set_index('modifier')[['relevance', 'groundedness', 'completeness', 
                                                  'actionability', 'tone_empathy', 'safety']]
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data * 100, annot=True, fmt='.1f', cmap='RdYlGn', 
            vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Pass Rate (%)'})
ax.set_title('Judge Pass Rates by Modifier (%)')
ax.set_ylabel('Modifier')
ax.set_xlabel('Judge')
plt.tight_layout()
plt.show()

## Worst-Performing Slices (n ≥ 30)

**Takeaway:** All slices with 30+ queries show 0% overall pass rate. The lowest performers tend to be verification/identity issues and non-native modifiers.

In [ ]:
# Find worst-performing slices with n >= 30
slices = []

# By persona
for persona in eval_df['persona_id'].unique():
    subset = eval_df[eval_df['persona_id'] == persona]
    if len(subset) >= 30:
        overall = subset[[f'{c}_passed' for c in criteria]].all(axis=1).mean()
        slices.append({
            'slice_type': 'persona',
            'slice_value': persona,
            'n': len(subset),
            'overall_pass_rate': overall
        })

# By problem
for problem in eval_df['problem_id'].unique():
    subset = eval_df[eval_df['problem_id'] == problem]
    if len(subset) >= 30:
        overall = subset[[f'{c}_passed' for c in criteria]].all(axis=1).mean()
        slices.append({
            'slice_type': 'problem',
            'slice_value': problem,
            'n': len(subset),
            'overall_pass_rate': overall
        })

# By modifier
for modifier in eval_df['modifier_id'].unique():
    subset = eval_df[eval_df['modifier_id'] == modifier]
    if len(subset) >= 30:
        overall = subset[[f'{c}_passed' for c in criteria]].all(axis=1).mean()
        slices.append({
            'slice_type': 'modifier',
            'slice_value': modifier,
            'n': len(subset),
            'overall_pass_rate': overall
        })

slices_df = pd.DataFrame(slices).sort_values('overall_pass_rate')
display(slices_df.head(15))

print("\nWorst-performing slices (n ≥ 30):")
for _, row in slices_df.head(10).iterrows():
    print(f"  {row['slice_type']}:{row['slice_value']} - {row['overall_pass_rate']:.2%} (n={row['n']})")

## Error Clustering: Groundedness (Worst Criterion)

**Takeaway:** Groundedness failures show patterns around: (1) generic advice that doesn't cite specific articles, (2) hallucinated features/limits not in retrieved context, (3) missing steps in multi-action workflows.

In [ ]:
# Analyze groundedness failures
failures = eval_df[~eval_df['groundedness_passed']].copy()

# Sample reasoning from failures
print(f"Groundedness failures: {len(failures)} / {len(eval_df)}")
print("\nSample failure reasoning:")
for i, row in failures.head(5).iterrows():
    print(f"\n--- Failure {i} ---")
    print(f"Query: {row['query'][:100]}...")
    print(f"Reasoning: {row['groundedness_reasoning']}")

# Common patterns in failures (keyword analysis)
reasoning_text = ' '.join(failures['groundedness_reasoning'].astype(str).tolist())
common_phrases = [
    'not mentioned',
    'not found', 
    'not cited',
    'generic',
    'hallucinat',
    'without reference',
    'not supported',
    'not retrieved'
]

phrase_counts = {}
for phrase in common_phrases:
    count = reasoning_text.lower().count(phrase)
    if count > 0:
        phrase_counts[phrase] = count

print("\nCommon failure patterns:")
for phrase, count in sorted(phrase_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  '{phrase}': {count} occurrences")

print(f"\nTotal groundedness failures analyzed: {len(failures)}")

## Error Clustering: Tone/Empathy (Near-Zero Performance)

**Takeaway:** Tone_empathy failures show the judge is looking for warmth, personalization, and emotional acknowledgment. Current answers are too factual/transactional. This judge may be over-strict (0.37% is unrealistically low).

In [ ]:
# Analyze tone/empathy failures
tone_failures = eval_df[~eval_df['tone_empathy_passed']].copy()
tone_passes = eval_df[eval_df['tone_empathy_passed']].copy()

print(f"Tone/empathy failures: {len(tone_failures)} / {len(eval_df)}")
print(f"Tone/empathy passes: {len(tone_passes)} / {len(eval_df)}")

if len(tone_passes) > 0:
    print("\nSample PASS reasoning (what worked):")
    for i, row in tone_passes.head(2).iterrows():
        print(f"\n--- Pass {i} ---")
        print(f"Answer: {row['answer'][:200]}...")
        print(f"Reasoning: {row['tone_empathy_reasoning']}")

print("\nSample FAILURE reasoning (what didn't work):")
for i, row in tone_failures.head(3).iterrows():
    print(f"\n--- Failure {i} ---")
    print(f"Answer snippet: {row['answer'][:150]}...")
    print(f"Reasoning: {row['tone_empathy_reasoning']}")

# Common patterns
reasoning_text = ' '.join(tone_failures['tone_empathy_reasoning'].astype(str).tolist())
common_phrases = [
    'not empathetic',
    'not warm',
    'robotic',
    'impersonal',
    ' lacks',
    'not personalized',
    'transactional',
    'formal'
]

phrase_counts = {}
for phrase in common_phrases:
    count = reasoning_text.lower().count(phrase)
    if count > 0:
        phrase_counts[phrase] = count

print("\nCommon failure patterns:")
for phrase, count in sorted(phrase_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  '{phrase}': {count} occurrences")

## Conclusions & Next Steps

**Takeaway:** The RAG system retrieves relevant articles (78% relevance) and provides complete information (74%), but suffers from critical groundedness issues (19%) and completely lacks human warmth (0.37% tone). 

### Key Findings:

1. **Overall Pass Rate: 0%** - No query passed all 6 judges

2. **Groundedness (19%)** - Main product issue:
   - RAG answers hallucinate or go beyond retrieved context
   - Generic advice without article citations
   - Missing steps in multi-action workflows

3. **Tone/Empathy (0.37%)** - Likely over-strict judge:
   - Current answers are factual/transactional
   - Judge expects warmth and emotional acknowledgment
   - 0.37% pass rate is unrealistically low → calibration question for Part 2

4. **Safety/Compliance (38%)** - Missing disclaimers and cautions

5. **Strong Areas:**
   - Relevance (78%) - retrieval works
   - Completeness (74%) - covers most information needs
   - Actionability (68%) - mostly usable instructions

### Part 2 Priority:
Focus GEPA optimization on **groundedness** as the main product weakness. **Tone/empathy** needs judge recalibration before prompt optimization.

In [ ]:
# Summary table for Part 2 planning
summary_df = pd.DataFrame([
    {'Criterion': 'Overall (all 6)', 'Pass Rate': '0%', 'Status': '❌ Critical', 'Part 2 Action': 'Focus on groundedness'},
    {'Criterion': 'Relevance', 'Pass Rate': '78%', 'Status': '✓ Strong', 'Part 2 Action': 'Maintain'},
    {'Criterion': 'Groundedness', 'Pass Rate': '19%', 'Status': '❌ Critical', 'Part 2 Action': 'GEPA optimization'},
    {'Criterion': 'Completeness', 'Pass Rate': '74%', 'Status': '✓ Strong', 'Part 2 Action': 'Maintain'},
    {'Criterion': 'Actionability', 'Pass Rate': '68%', 'Status': '~ Moderate', 'Part 2 Action': 'Monitor'},
    {'Criterion': 'Tone/Empathy', 'Pass Rate': '0.4%', 'Status': '? Calibration', 'Part 2 Action': 'Recalibrate judge'},
    {'Criterion': 'Safety/Compliance', 'Pass Rate': '38%', 'Status': '⚠️ Weak', 'Part 2 Action': 'Secondary priority'}
])

display(summary_df)

print("\nPart 2 Cost Estimate:")
print("  GEPA optimization on 200 weakest queries (groundedness focus):")
print("  - Gold labels (gpt-4o): 200 calls × $0.005 = $1.00")
print("  - GEPA evolution (50 iter × 3 candidates): 150 calls × $0.001 = $0.15")
print("  - Total Part 2: ~$1.15")
print("\n✓ Part 1 RAG evaluation complete")
print("✓ Ready for Part 2: GEPA optimization on groundedness")